# Learning on Graphs: Homework 2

## Spatio-Temporal Forecasting on Graphs

This notebook accompanies the assignment PDF. Complete the cells marked `TODO`, keep the default settings unless the PDF asks otherwise, and submit the completed notebook together with the report.


In [ ]:
import math
import random
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

# Keep numerical experiments lightweight and reproducible across machines.
torch.set_num_threads(1)


In [ ]:
def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(0)


## Part A: temporal graph dataset and forecasting windows

The functions below are provided. Use them to generate the dataset, visualize it, and construct chronological forecasting windows.


In [ ]:
def make_sbm_graph(seed=0, sizes=(30, 30), probs=None, largest_component=True):
    if probs is None:
        probs = [[0.18, 0.02], [0.02, 0.18]]
    graph = nx.stochastic_block_model(list(sizes), probs, seed=seed)
    if largest_component and not nx.is_connected(graph):
        largest = max(nx.connected_components(graph), key=len)
        graph = graph.subgraph(largest).copy()
        graph = nx.convert_node_labels_to_integers(graph)
    return graph


def normalized_adjacency(graph):
    adjacency = nx.to_numpy_array(graph, dtype=np.float32)
    adjacency = adjacency + np.eye(adjacency.shape[0], dtype=np.float32)
    degree = adjacency.sum(axis=1)
    degree_inv_sqrt = np.power(degree, -0.5)
    degree_inv_sqrt[~np.isfinite(degree_inv_sqrt)] = 0.0
    return (degree_inv_sqrt[:, None] * adjacency * degree_inv_sqrt[None, :]).astype(np.float32)


def graph_stats(graph):
    degrees = np.array([degree for _, degree in graph.degree()], dtype=float)
    return {
        "num_nodes": graph.number_of_nodes(),
        "num_edges": graph.number_of_edges(),
        "average_degree": float(degrees.mean()),
        "max_degree": int(degrees.max()),
        "is_connected": nx.is_connected(graph),
    }


In [ ]:
def make_forcing(num_nodes, total_steps, source_fraction=0.12, seed=0):
    rng = np.random.default_rng(seed)
    num_sources = max(1, int(source_fraction * num_nodes))
    source_nodes = rng.choice(num_nodes, size=num_sources, replace=False)
    forcing = np.zeros((total_steps, num_nodes), dtype=np.float32)

    time = np.arange(total_steps, dtype=np.float32)
    for node in source_nodes:
        frequency = rng.uniform(0.03, 0.08)
        phase = rng.uniform(0.0, 2.0 * np.pi)
        amplitude = rng.uniform(0.7, 1.3)
        forcing[:, node] = amplitude * np.sin(frequency * time + phase)
    return forcing, source_nodes


def simulate_temporal_signal(
    normalized_adj,
    total_steps=400,
    alpha=0.45,
    beta=0.35,
    gamma=0.15,
    noise_std=0.03,
    seed=0,
):
    rng = np.random.default_rng(seed)
    num_nodes = normalized_adj.shape[0]
    forcing, source_nodes = make_forcing(num_nodes, total_steps, seed=seed)
    signals = np.zeros((total_steps, num_nodes), dtype=np.float32)
    signals[0] = rng.normal(0.0, 0.2, size=num_nodes).astype(np.float32)

    for time_index in range(total_steps - 1):
        noise = rng.normal(0.0, noise_std, size=num_nodes).astype(np.float32)
        signals[time_index + 1] = (
            alpha * (normalized_adj @ signals[time_index])
            + beta * signals[time_index]
            + gamma * forcing[time_index]
            + noise
        )
    return signals, forcing, source_nodes


In [ ]:
def plot_graph(graph, values=None, position=None, title="Graph", vmin=None, vmax=None):
    if position is None:
        position = nx.spring_layout(graph, seed=0)
    plt.figure(figsize=(5, 4))
    if values is None:
        nx.draw_networkx(
            graph,
            pos=position,
            node_size=80,
            with_labels=False,
            edge_color="0.75",
            node_color="tab:blue",
        )
    else:
        nx.draw_networkx_edges(graph, position, edge_color="0.80", width=0.8)
        nodes = nx.draw_networkx_nodes(
            graph,
            position,
            node_color=values,
            node_size=80,
            cmap="coolwarm",
            vmin=vmin,
            vmax=vmax,
        )
        plt.colorbar(nodes, fraction=0.046, pad=0.04)
    plt.title(title)
    plt.axis("off")
    plt.show()
    return position


def plot_node_series(signals, node_indices, title="Selected node signals"):
    plt.figure(figsize=(8, 3))
    for node in node_indices:
        plt.plot(signals[:, node], label=f"node {node}")
    plt.xlabel("time")
    plt.ylabel("signal value")
    plt.title(title)
    plt.legend(ncol=2, fontsize=8)
    plt.tight_layout()
    plt.show()


def plot_loss_curves(history, title="Training curves"):
    plt.figure(figsize=(5, 3))
    plt.plot(history["train"], label="train")
    plt.plot(history["val"], label="validation")
    plt.xlabel("epoch")
    plt.ylabel("MSE")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# TODO A1: generate the default graph and temporal signals.
# Required objects: G, A_hat, X, U, source_nodes, train_end, val_end.

set_seed(0)
G = None
A_hat = None
X = None
U = None
source_nodes = None
train_end, val_end = None, None

# Print graph_stats(G) and the split ranges after completing this cell.


In [ ]:
# TODO A2: create the required figures:
# 1. graph visualization,
# 2. time series for 5 selected nodes,
# 3. graph-signal snapshots at 3 time points.


In [ ]:
def make_windows(signals, window_size, train_end, validation_end):
    # Construct chronological forecasting windows.
    # Returns a dictionary with keys train, val, test.
    # Each value is (inputs, targets, target_times), where
    #   inputs has shape num_examples x num_nodes x window_size,
    #   targets has shape num_examples x num_nodes.
    # The target for a window ending at time t is x_{t+1}.
    # TODO A3: implement this function.
    # Hint: loop over end_time from window_size - 1 to total_steps - 2.
    # target_time = end_time + 1.
    # Assign the example according to target_time.
    raise NotImplementedError

# TODO A3: construct windows for L=5 and report split sizes.


## Part B: forecasting baselines

Implement the persistence baseline, a node-wise temporal MLP, and the linear diffusion baseline.


In [ ]:
def mse_np(prediction, target):
    return float(np.mean((prediction - target) ** 2))


def evaluate_persistence(windows):
    # Persistence baseline: predict x_{t+1} = x_t.
    # TODO B1: implement validation and test evaluation.
    # For an input window H, the last column H[:, :, -1] is x_t.
    raise NotImplementedError


In [ ]:
def to_tensor(array):
    return torch.tensor(array, dtype=torch.float32, device=DEVICE)


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


class TemporalMLP(nn.Module):
    def __init__(self, window_size, hidden_dim=32):
        super().__init__()
        # TODO B2: define a node-wise MLP mapping a length-L history to one scalar.
        # Suggested structure: Linear(window_size, hidden_dim), ReLU, Linear(hidden_dim, 1).
        raise NotImplementedError

    def forward(self, history_window, graph_matrix=None):
        # history_window: batch x nodes x L
        # return: batch x nodes
        raise NotImplementedError


In [ ]:
def train_model(model, train_data, val_data, graph_matrix=None, epochs=30, lr=1e-2, weight_decay=0.0):
    train_inputs, train_targets, _ = train_data
    val_inputs, val_targets, _ = val_data

    train_inputs = to_tensor(train_inputs)
    train_targets = to_tensor(train_targets)
    val_inputs = to_tensor(val_inputs)
    val_targets = to_tensor(val_targets)
    graph_tensor = None if graph_matrix is None else to_tensor(graph_matrix)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_state = None
    best_val = float("inf")
    history = {"train": [], "val": []}

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        train_prediction = model(train_inputs, graph_tensor)
        train_loss = F.mse_loss(train_prediction, train_targets)
        train_loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_prediction = model(val_inputs, graph_tensor)
            val_loss = F.mse_loss(val_prediction, val_targets)

        train_value = float(train_loss.detach().cpu())
        val_value = float(val_loss.detach().cpu())
        history["train"].append(train_value)
        history["val"].append(val_value)

        if val_value < best_val:
            best_val = val_value
            best_state = {name: parameter.detach().cpu().clone() for name, parameter in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return history, best_val


def evaluate_model(model, data, graph_matrix=None):
    inputs, targets, _ = data
    model.eval()
    graph_tensor = None if graph_matrix is None else to_tensor(graph_matrix)
    with torch.no_grad():
        prediction = model(to_tensor(inputs), graph_tensor)
        loss = F.mse_loss(prediction, to_tensor(targets))
    return float(loss.detach().cpu())


In [ ]:
def fit_linear_diffusion(windows, graph_matrix):
    # Fit x_{t+1} = a x_t + b A_hat x_t + c by least squares on training windows.
    # TODO B3: fit a, b, c using all nodes and all training windows.
    raise NotImplementedError


def predict_linear_diffusion(inputs, graph_matrix, coefficients):
    # TODO B3: apply fitted coefficients to a batch of windows.
    raise NotImplementedError


def evaluate_linear_diffusion(windows, graph_matrix, coefficients):
    # TODO B3: return validation and test MSE.
    raise NotImplementedError


In [ ]:
# TODO B: run the persistence, TemporalMLP, and linear diffusion baselines.
# Store validation and test MSE values for the report tables.


## Part C: GCN and spatio-temporal GNN models

Implement the dense GCN layer, a GCN forecaster, and the spatio-temporal GNN described in the PDF.


In [ ]:
class DenseGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        # TODO C1: define parameters for H -> A_hat H W + b.
        raise NotImplementedError

    def forward(self, node_features, graph_matrix):
        # node_features: batch x nodes x in_dim
        # graph_matrix: nodes x nodes
        # return: batch x nodes x out_dim
        raise NotImplementedError


class GCNForecaster(nn.Module):
    def __init__(self, hidden_dim=32):
        super().__init__()
        # TODO C2: two DenseGCNLayer modules.
        # This model uses only the last signal x_t from the input window.
        raise NotImplementedError

    def forward(self, history_window, graph_matrix):
        # history_window: batch x nodes x L
        # Use history_window[:, :, -1:] as the current signal.
        raise NotImplementedError


In [ ]:
class SpatioTemporalGNN(nn.Module):
    def __init__(self, window_size, hidden_dim=32, kernel_size=3):
        super().__init__()
        # TODO C3:
        # 1. temporal Conv1d from 1 channel to hidden_dim channels,
        # 2. DenseGCNLayer hidden_dim -> hidden_dim,
        # 3. DenseGCNLayer hidden_dim -> 1.
        # Use padding=1 for kernel_size=3.
        raise NotImplementedError

    def forward(self, history_window, graph_matrix):
        # history_window: batch x nodes x L
        # Step 1: reshape to (batch * nodes) x 1 x L.
        # Step 2: temporal convolution + ReLU.
        # Step 3: mean-pool over the time dimension to obtain batch x nodes x hidden_dim.
        # Step 4: graph convolution, ReLU, graph convolution.
        # Return batch x nodes.
        raise NotImplementedError


In [ ]:
# TODO C4: train and evaluate
# 1. GCNForecaster using L=5 windows,
# 2. SpatioTemporalGNN using L=5 windows.
# Plot training curves and report parameter counts, validation MSE, and test MSE.


## Part D: ablations and final comparison

Run the history-length and graph ablations, then collect the final tables for the report.


In [ ]:
# TODO D1: history-length ablation for the SpatioTemporalGNN.
# Train with L in {1, 3, 5, 10}. Rebuild the windows for each L.


In [ ]:
# TODO D2: graph ablation.
# Repeat the L=5 SpatioTemporalGNN experiment with graph_matrix = identity.


In [ ]:
# TODO D3: collect final results in tables for the report.
# Include the baselines, GCNForecaster, best-L SpatioTemporalGNN, and no-graph ablation.
